# SpendDNA – Personal Spending Analysis

**Name:** Devipriya S  
**Batch:** Data Science/Data Analytics July 2026 Batch

**Date:** 09 August 2026

A Python-based personal finance analysis project that processes transaction data
and identifies spending patterns, trends, anomalies, and spending archetypes.

In [91]:
import numpy as np
import pandas as pd

In [92]:
data=pd.read_csv("/content/12980098-DADS_MP2_Dataset.zip")

In [93]:
data.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [94]:
data.shape

(1328, 8)

In [95]:
data.columns

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')

In [96]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB



## FEATURE 1 : TRANSACTION PARSER


In [97]:
# AI-assisted: Helped implement parsing for mixed date formats using Pandas.
data["Date"] = pd.to_datetime(
    data["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [98]:
# AI-assisted: Helped implement cleaning and conversion of mixed currency formats
data["Amount"] = data["Amount"].astype(str)

data["Amount"] = (
    data["Amount"]
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

data["Amount"] = pd.to_numeric(data["Amount"], errors="coerce")

In [99]:
data["Type"] = data["Type"].replace({
    "DR": "Debit",
    "CR": "Credit"
})

data["Type"] = data["Type"].str.lower()

In [100]:
data["Mode"] = data["Mode"].replace("", np.nan)

In [101]:
before = len(data)

data = data.drop_duplicates()

after = len(data)

duplicates = before - after

In [102]:
data = data.dropna(subset=["Date","Amount"])

In [103]:
print("="*60)
print("FEATURE 1 : TRANSACTION PARSER")
print("="*60)

print(
    f"Parsed {len(data)} transactions across "
    f"{data['Date'].dt.month.nunique()} months."
)
print(f"Dropped Duplicates  : {duplicates}")
print(f"Unparseable dates   : {data['Date'].isna().sum()}")
print(f"Unparseable amounts : {data['Amount'].isna().sum()}")

FEATURE 1 : TRANSACTION PARSER
Parsed 1310 transactions across 6 months.
Dropped Duplicates  : 18
Unparseable dates   : 0
Unparseable amounts : 0



##FEATURE 2 : VENDOR EXTRACTOR


In [104]:
# Inspect all unique descriptions
for desc in sorted(data["Description"].unique()):
    print(desc)

AIRTEL POSTPAID
AMAZON IN
AMAZON PRIME VIDEO
AMAZON SELLER SVCS
AMAZONIN MARKETPLACE
AMZN PRIME
AMZN-INTPYMT
ANI Technologies
ATM-WDL-HDFC-3609
ATM-WDL-HDFC-4942
ATM-WDL-HDFC-8030
ATM-WDL-HDFC-8253
ATM-WDL-HDFC-9140
ATM-WDL-ICICI-3918
ATM-WDL-ICICI-4172
ATM-WDL-ICICI-4739
ATM-WDL-ICICI-5025
ATM-WDL-ICICI-6478
ATM-WDL-ICICI-9135
ATM-WDL-SBI-0237
ATM-WDL-SBI-0279
ATM-WDL-SBI-0874
ATM-WDL-SBI-4080
ATM-WDL-SBI-4084
ATM-WDL-SBI-5715
AVENUE SUPERMARTS
Amazon Pay India
BANGALORE ELEC SUPPLY
BESCOM ELEC BILL
BHARTI AIRTEL LTD
BHIM SWIGGY
BHIM ZEPTO
BHIM-BLINKIT
BHIM-BMTC
BIGBASKET BANGALORE
BIGTREE ENTERTAINMENT
BLINKIT BANGALORE
BMS MOVIE TICKETS
BMTC BUS PASS
BUNDL TECH-INSTAMART
BUNDL Tech P L
BWSSB WATER BILL
COFFEE DAY GLOBAL
DISNEY HOTSTAR
FKART INTRNET
FLIPKART INDIA
FSN E-COMMERCE
Flipkart Internet
GROFERS INDIA P L
GROWW INVEST TECH
IMPS ZERODHA-COIN
IMPS-RENT-LANDLORD-35126704
IMPS-RENT-LANDLORD-36852906
IMPS-RENT-LANDLORD-39598076
IMPS-RENT-LANDLORD-49966195
IMPS-RENT-LANDLORD-51148

In [105]:
  # AI-assisted: Helped structure the vendor keyword dictionary for merchant normalization.
vendor_dict = {
    "Swiggy": [
        "SWIGGY",
        "BUNDL"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN",
        "AMAZONPAY"
    ],

    "Zepto": [
        "ZEPTO"
    ],

    "Blinkit": [
        "BLINKIT",
        "GROFERS"
    ],

    "BigBasket": [
        "BIGBASKET"
    ],

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA"
    ],

    "BookMyShow": [
        "BOOKMYSHOW"
    ],

    "Netflix":[
        "NETFLIX"
    ],

    "Spotify":[
        "SPOTIFY"
    ],

    "Hotstar":[
        "HOTSTAR"
    ],

    "Prime":[
        "PRIME"
    ],

    "Zerodha":[
        "ZERODHA",
        "COIN"
    ],

    "Groww":[
        "GROWW"
    ]
}

In [106]:
 # AI-assisted: Helped design the conditional vendor extraction logic for messy descriptions.
def extract_vendor(desc):

    desc = str(desc).upper()

    # ---------------- FOOD DELIVERY ----------------
    if any(x in desc for x in [
        "SWIGGY",
        "BUNDL"
    ]):
        return "Swiggy"

    if "ZOMATO" in desc:
        return "Zomato"

    # ---------------- QUICK COMMERCE ----------------
    if any(x in desc for x in [
        "ZEPTO",
        "BHIM ZEPTO",
        "ZEPTO MARKETPLACE"
    ]):
        return "Zepto"

    if any(x in desc for x in [
        "BLINKIT",
        "GROFERS"
    ]):
        return "Blinkit"

    if any(x in desc for x in [
        "BIGBASKET",
        "KIRANAKART"
    ]):
        return "BigBasket"

    if "INSTAMART" in desc:
        return "Instamart"

    # ---------------- SHOPPING ----------------
    if any(x in desc for x in [
        "AMAZON",
        "AMZN",
        "AMAZON PAY"
    ]):
        return "Amazon"

    if any(x in desc for x in [
        "FLIPKART",
        "FKART"
    ]):
        return "Flipkart"

    if "MYNTRA" in desc:
        return "Myntra"

    if "NYKAA" in desc:
        return "Nykaa"

    if any(x in desc for x in [
        "DMART",
        "AVENUE SUPERMARTS"
    ]):
        return "DMart"

    # ---------------- TRANSPORT ----------------
    if "UBER" in desc:
        return "Uber"

    if "OLA" in desc:
        return "Ola"

    if "RAPIDO" in desc:
        return "Rapido"

    if any(x in desc for x in [
        "BMTC",
        "TUMMOC"
    ]):
        return "BMTC"

    if "ROPPEN" in desc:
        return "Metro"

    # ---------------- INVESTMENTS ----------------
    if any(x in desc for x in [
        "ZERODHA",
        "COIN"
    ]):
        return "Zerodha"

    if "GROWW" in desc:
        return "Groww"

    # ---------------- ENTERTAINMENT ----------------
    if any(x in desc for x in [
        "BOOKMYSHOW",
        "BIGTREE",
        "BMS"
    ]):
        return "BookMyShow"

    # ---------------- SUBSCRIPTIONS ----------------
    if "NETFLIX" in desc:
        return "Netflix"

    if "SPOTIFY" in desc:
        return "Spotify"

    if any(x in desc for x in [
        "HOTSTAR",
        "STAR INDIA"
    ]):
        return "Hotstar"

    if any(x in desc for x in [
        "AMZN PRIME",
        "PRIME VIDEO"
    ]):
        return "Prime Video"

    # ---------------- CAFE ----------------
    if any(x in desc for x in [
        "STARBUCKS",
        "TATA STARBUCKS"
    ]):
        return "Starbucks"

    if any(x in desc for x in [
        "COFFEE DAY",
        "CCD"
    ]):
        return "Cafe Coffee Day"

    if any(x in desc for x in [
        "THIRD WAVE",
        "TWC INDIA"
    ]):
        return "Third Wave Coffee"

    # ---------------- RESTAURANTS ----------------
    if "EMPIRE" in desc:
        return "Empire"

    if "MEGHANA" in desc:
        return "Meghana"

    if "TRUFFLES" in desc:
        return "Truffles"

    if any(x in desc for x in [
        "RESTAURANT",
        "DINEOUT"
    ]):
        return "Restaurant"

    # ---------------- UTILITIES ----------------
    if any(x in desc for x in [
        "BESCOM",
        "ELEC SUPPLY"
    ]):
        return "BESCOM"

    if "BWSSB" in desc:
        return "BWSSB"

    if "AIRTEL" in desc:
        return "Airtel"

    if any(x in desc for x in [
        "JIO",
        "JIOFIBER"
    ]):
        return "Jio"

    if any(x in desc for x in [
        "VI ",
        "VODAFONE"
    ]):
        return "Vi"

    # ---------------- FUEL ----------------
    if any(x in desc for x in [
        "INDIAN OIL",
        "IOC"
    ]):
        return "IndianOil"

    if "HP PETROL" in desc:
        return "HPCL"

    if "BPCL" in desc:
        return "BPCL"

    # ---------------- INCOME ----------------
    if "SALARY" in desc:
        return "Salary"

    # ---------------- RENT ----------------
    if "LANDLORD" in desc:
        return "Rent"

    # ---------------- CASH ----------------
    if "ATM-WDL" in desc:
        return "Cash Withdrawal"

    # ---------------- PERSONAL TRANSFER ----------------
    if any(x in desc for x in [
        "AMAN",
        "ANKIT",
        "PRIYA",
        "NEHA",
        "SNEHA",
        "KARAN",
        "VIKAS"
    ]):
        return "Personal Transfer"


    return "Others"

In [107]:
# AI-assisted: Helped apply the vendor extraction function to transaction descriptions.

data["vendor_clean"] = data["Description"].apply(extract_vendor)

In [108]:
print("Canonical Vendors Identified :", data["vendor_clean"].nunique())

print("\nVendor Frequency\n")

print(
    data["vendor_clean"]
    .value_counts()
)

Canonical Vendors Identified : 42

Vendor Frequency

vendor_clean
Swiggy               223
Zomato               121
Amazon                86
Uber                  71
Ola                   69
Zepto                 58
Blinkit               55
Flipkart              47
Others                43
Starbucks             42
Rapido                41
Restaurant            39
BMTC                  37
Cafe Coffee Day       26
BigBasket             24
DMart                 22
Instamart             20
Myntra                20
Third Wave Coffee     20
Personal Transfer     18
IndianOil             17
Cash Withdrawal       17
Nykaa                 16
BESCOM                15
Zerodha               14
Metro                 14
BookMyShow            13
Hotstar               13
Empire                13
Meghana               12
Netflix               10
Groww                  9
HPCL                   9
Truffles               9
Spotify                8
BWSSB                  8
Salary                 6
Jio      

In [109]:
print("="*60)
print("FEATURE 2 : VENDOR EXTRACTOR")
print("="*60)
print(f"Canonical Vendors Identified : {data['vendor_clean'].nunique()}")

print("\nTop Vendors\n")
print(data["vendor_clean"].value_counts().head(10))

FEATURE 2 : VENDOR EXTRACTOR
Canonical Vendors Identified : 42

Top Vendors

vendor_clean
Swiggy       223
Zomato       121
Amazon        86
Uber          71
Ola           69
Zepto         58
Blinkit       55
Flipkart      47
Others        43
Starbucks     42
Name: count, dtype: int64



##Feature 3 - Category Tagger


In [110]:
# AI-assisted: Helped structure the vendor-to-category mapping used for spending analysis.
category_dict = {

    "Swiggy":"Food Delivery",
    "Zomato":"Food Delivery",

    "Blinkit":"Quick Commerce",
    "Zepto":"Quick Commerce",
    "BigBasket":"Quick Commerce",
    "Instamart":"Quick Commerce",

    "Amazon":"E-commerce",
    "Flipkart":"E-commerce",
    "Myntra":"E-commerce",
    "Nykaa":"E-commerce",
    "DMart":"Groceries",

    "Uber":"Transport",
    "Ola":"Transport",
    "Rapido":"Transport",
    "BMTC":"Transport",
    "Metro":"Transport",

    "Restaurant":"Restaurants",
    "Empire":"Restaurants",
    "Meghana":"Restaurants",
    "Truffles":"Restaurants",

    "Starbucks":"Cafe",
    "Cafe Coffee Day":"Cafe",
    "Third Wave Coffee":"Cafe",

    "BookMyShow":"Entertainment",

    "Netflix":"Subscriptions",
    "Spotify":"Subscriptions",
    "Prime Video":"Subscriptions",
    "Hotstar":"Subscriptions",

    "IndianOil":"Fuel",
    "HPCL":"Fuel",
    "BPCL":"Fuel",

    "BESCOM":"Utilities",
    "BWSSB":"Utilities",
    "Airtel":"Utilities",
    "Jio":"Utilities",
    "Vi":"Utilities",

    "Groww":"Investments",
    "Zerodha":"Investments",

    "Salary":"Income",
    "Rent":"Housing",

    "Cash Withdrawal":"Cash Withdrawal",
    "Personal Transfer":"Personal Transfer"
}

In [111]:
data["Category"] = data["vendor_clean"].map(category_dict)
data["Category"] = data["Category"].fillna("Others")

In [112]:
print("="*60)
print("FEATURE 3 : CATEGORY TAGGER")
print("="*60)

print("\nCategory Frequency\n")
print(data["Category"].value_counts())

FEATURE 3 : CATEGORY TAGGER

Category Frequency

Category
Food Delivery        344
Transport            232
E-commerce           169
Quick Commerce       157
Cafe                  88
Restaurants           73
Others                43
Utilities             40
Subscriptions         31
Fuel                  28
Investments           23
Groceries             22
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Housing                6
Name: count, dtype: int64



##Feature 4 - Spending Overview


In [113]:
 # AI-assisted: Helped structure the calculations for credits, debits, net savings, and savings rate
print("="*60)
print("FEATURE 4 : SPENDING OVERVIEW")
print("="*60)

credits = data[data["Type"] == "credit"]["Amount"].sum()
debits = data[data["Type"] == "debit"]["Amount"].sum()

net = credits - debits

saving_rate = (net / credits) * 100

print(f"Total Credits      : ₹{credits:,.2f}")
print(f"Total Debits       : ₹{debits:,.2f}")
print(f"Net Savings        : ₹{net:,.2f}")
print(f"Savings Rate       : {saving_rate:.2f}%")
print(f"Financial Status   : {'OVER-SPENDER' if saving_rate < 0 else 'SAVER'}")
print(f"Total Transactions : {len(data)}")

FEATURE 4 : SPENDING OVERVIEW
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Financial Status   : OVER-SPENDER
Total Transactions : 1310


In [114]:
print("\nTop 5 Categories by Spend")
print(
    data[data["Type"] == "debit"]
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)


Top 5 Categories by Spend
Category
E-commerce       599581.0
Investments      248160.0
Food Delivery    150839.0
Restaurants      117737.0
Housing          108000.0
Name: Amount, dtype: float64


In [115]:
print("\nTop 5 Vendors by Spend")
print(
    data[data["Type"] == "debit"]
    .groupby("vendor_clean")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)


Top 5 Vendors by Spend
vendor_clean
Amazon      328530.0
Zerodha     210000.0
Flipkart    177510.0
Rent        108000.0
Swiggy       95523.0
Name: Amount, dtype: float64



##Feature 5 - Monthly Trend Analysis

In [116]:
 # AI-assisted: Helped structure monthly spending aggregation and chronological month ordering.
print("="*60)
print("FEATURE 5 : MONTHLY TREND ANALYSIS")
print("="*60)

# Create Month column
data["Month"] = data["Date"].dt.month_name()

# Set correct chronological month order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

data["Month"] = pd.Categorical(
    data["Month"],
    categories=month_order,
    ordered=True
)

# Monthly spending by category
monthly_trend = data[data["Type"] == "debit"].pivot_table(
    values="Amount",
    index="Category",
    columns="Month",
    aggfunc="sum",
    fill_value=0,
    observed=False
)

# Arrange columns chronologically
monthly_trend = monthly_trend.reindex(columns=month_order)

print("\nMonthly Spending by Category:\n")
print(monthly_trend)

FEATURE 5 : MONTHLY TREND ANALYSIS

Monthly Spending by Category:

Month              January  February     March    April      May      June
Category                                                                  
Cafe                2824.0    4044.0    4521.0   5826.0   5668.0    5056.0
Cash Withdrawal     2000.0    5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         98623.0   92738.0  105977.0  69219.0  95776.0  137248.0
Entertainment       1263.0     474.0    2418.0   2224.0      0.0    1914.0
Food Delivery      22633.0   23740.0   24803.0  27756.0  25408.0   26499.0
Fuel               30322.0    2079.0   26164.0  18718.0   9138.0    2882.0
Groceries          14594.0    3873.0    3658.0   3334.0   7227.0    4777.0
Housing            18000.0   18000.0   18000.0  18000.0  18000.0   18000.0
Investments        38432.0   15000.0   68644.0  54126.0  48628.0   23330.0
Others              1133.0    2720.0    5154.0   8455.0   3544.0    3341.0
Personal Transfer   7852.0    428

In [117]:
# AI-assisted: Helped implement percentage-based monthly growth and decline calculations.
# Calculate percentage change from January to June
first_month = monthly_trend["January"]
last_month = monthly_trend["June"]

growth = ((last_month - first_month) / first_month) * 100

# Remove categories where January spending is zero
growth = growth.replace([np.inf, -np.inf], np.nan).dropna()

# Biggest growth
biggest_growth_category = growth.idxmax()
biggest_growth_value = growth.max()

# Biggest decline
biggest_decline_category = growth.idxmin()
biggest_decline_value = growth.min()

print("\nTrend Analysis:")
print(
    f"Biggest Growth   : {biggest_growth_category} "
    f"({biggest_growth_value:.2f}%)"
)

print(
    f"Biggest Decline  : {biggest_decline_category} "
    f"({biggest_decline_value:.2f}%)"
)


Trend Analysis:
Biggest Growth   : Cash Withdrawal (750.00%)
Biggest Decline  : Fuel (-90.50%)



##Feature 6 - Time-of-Day Patterns


In [118]:
 # AI-assisted: Helped implement extraction of the transaction hour from the Time column.
# Extract Hour
data["Hour"] = data["Time"].str[:2].astype(int)
debit_data = data[data["Type"] == "debit"]

time_pattern = (
    debit_data
    .groupby(["Category", "Hour"])["Amount"]
    .sum()
    .unstack(fill_value=0)
)

In [119]:
# AI-assisted: Helped identify and format peak spending hours by category.
# Find peak spending hour for each category
peak_hours = time_pattern.idxmax(axis=1)
peak_amounts = time_pattern.max(axis=1)

print("\nPeak Spending Hour by Category:\n")

for category in peak_hours.index:
    print(
        f"{category:<20} "
        f"Hour: {peak_hours[category]:02d}:00 "
        f"Amount: ₹{peak_amounts[category]:,.2f}"
    )


Peak Spending Hour by Category:

Cafe                 Hour: 10:00 Amount: ₹3,637.00
Cash Withdrawal      Hour: 09:00 Amount: ₹10,500.00
E-commerce           Hour: 14:00 Amount: ₹72,584.00
Entertainment        Hour: 16:00 Amount: ₹1,881.00
Food Delivery        Hour: 20:00 Amount: ₹18,372.00
Fuel                 Hour: 15:00 Amount: ₹11,937.00
Groceries            Hour: 03:00 Amount: ₹7,080.00
Housing              Hour: 13:00 Amount: ₹54,000.00
Investments          Hour: 10:00 Amount: ₹90,000.00
Others               Hour: 10:00 Amount: ₹6,215.00
Personal Transfer    Hour: 17:00 Amount: ₹3,586.00
Quick Commerce       Hour: 09:00 Amount: ₹8,538.00
Restaurants          Hour: 20:00 Amount: ₹27,663.00
Subscriptions        Hour: 04:00 Amount: ₹4,685.00
Transport            Hour: 20:00 Amount: ₹4,380.00
Utilities            Hour: 18:00 Amount: ₹5,118.00


In [120]:
food_delivery = debit_data[
    debit_data["Category"] == "Food Delivery"
]

late_night = food_delivery[
    food_delivery["Hour"].isin([21, 22, 23, 0, 1, 2])
]

late_night_pct = (
    len(late_night) / len(food_delivery) * 100
)

print(
    f"Food Delivery orders between 9 PM and 2 AM: "
    f"{late_night_pct:.2f}%"
)

Food Delivery orders between 9 PM and 2 AM: 21.22%


In [121]:
print("="*60)
print("FEATURE 6 : TIME-OF-DAY PATTERNS")
print("="*60)

print("\nCategory × Hour Spending Matrix:\n")
print(time_pattern)

print("\nPeak Spending Hour by Category:\n")

for category in peak_hours.index:
    print(
        f"{category:<20} "
        f"{peak_hours[category]:02d}:00 "
        f"₹{peak_amounts[category]:,.2f}"
    )

FEATURE 6 : TIME-OF-DAY PATTERNS

Category × Hour Spending Matrix:

Hour                    0        1        2        3        4        5   \
Category                                                                  
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce         18650.0   9016.0  10246.0  14865.0  12018.0  17258.0   
Entertainment        562.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery        832.0   3017.0    928.0   1702.0   2618.0   3488.0   
Fuel                2675.0   2048.0      0.0    676.0   2010.0   2105.0   
Groceries           3895.0   1536.0   1717.0   7080.0   1169.0      0.0   
Housing                0.0      0.0      0.0      0.0      0.0      0.0   
Investments         4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Others                 0.0    785.0      0.0   2984.0    654.0    692.0   
Personal Transfer      0.0      


##Feature 7 - Anomaly Detection


In [122]:
# AI-assisted: Helped implement category-wise z-score calculation and anomaly filtering.
print("="*60)
print("FEATURE 7 : ANOMALY DETECTION")
print("="*60)

# Only debit transactions
debits = data[data["Type"] == "debit"].copy()

# Category-wise mean
category_mean = debits.groupby("Category")["Amount"].transform("mean")

# Category-wise standard deviation
category_std = debits.groupby("Category")["Amount"].transform("std")

# Calculate z-score
debits["z_score"] = (
    (debits["Amount"] - category_mean) / category_std
)

# Find anomalies
anomalies = debits[debits["z_score"] > 2].copy()

# Sort by z-score
anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print("\nTotal Anomalies:", len(anomalies))

print("\nTop 5 Anomalies:\n")

print(
    anomalies[
        ["Date", "vendor_clean", "Category", "Amount", "z_score"]
    ].head(5)
)

FEATURE 7 : ANOMALY DETECTION

Total Anomalies: 37

Top 5 Anomalies:

           Date vendor_clean        Category   Amount   z_score
522  2024-03-12    BigBasket  Quick Commerce   1865.0  4.613286
160  2024-01-23    BigBasket  Quick Commerce   1797.0  4.374408
1298 2024-06-26       Amazon      E-commerce  22008.0  4.054513
269  2024-02-07       Amazon      E-commerce  21986.0  4.049681
414  2024-02-26   Restaurant     Restaurants   8383.0  3.884639


##Feature 8 - Spending Archetype Detection


In [123]:
# AI-assisted: Helped structure the quantitative rules used to identify spending archetypes.
print("="*60)
print("FEATURE 8 : SPENDING ARCHETYPE DETECTION")
print("="*60)

# Total debit spending
total_debit = data[data["Type"] == "debit"]["Amount"].sum()

# Category spending
food = data[
    data["Category"].isin(
        ["Food Delivery", "Restaurants", "Cafe"]
    )
]["Amount"].sum()

quick = data[
    data["Category"] == "Quick Commerce"
]["Amount"].sum()

ecommerce = data[
    data["Category"] == "E-commerce"
]["Amount"].sum()

investment = data[
    data["Category"] == "Investments"
]["Amount"].sum()

# Percentages
food_pct = (food / total_debit) * 100
quick_pct = (quick / total_debit) * 100
ecommerce_pct = (ecommerce / total_debit) * 100
investment_pct = (investment / total_debit) * 100

print("\nSPENDING ARCHETYPES\n")

# Foodie
if food_pct > 25:
    print(
        f"THE FOODIE "
        f"({food_pct:.2f}% on Food Delivery/Restaurants/Cafe)"
    )

# Quick Commerce Junkie
if quick_pct > 15:
    print(
        f"THE QUICK COMMERCE JUNKIE "
        f"({quick_pct:.2f}% on Quick Commerce)"
    )

# Shopaholic
if ecommerce_pct > 15:
    print(
        f"THE SHOPAHOLIC "
        f"({ecommerce_pct:.2f}% on E-commerce)"
    )

# Investor
if investment_pct > 15:
    print(
        f"THE INVESTOR "
        f"({investment_pct:.2f}% on Investments)"
    )

# YOLO Spender
if saving_rate < 10:
    print(
        f"THE YOLO SPENDER "
        f"(Savings Rate: {saving_rate:.2f}%)"
    )

# Disciplined Saver
if saving_rate >= 40:
    print(
        f"THE DISCIPLINED SAVER "
        f"(Savings Rate: {saving_rate:.2f}%)"
    )

FEATURE 8 : SPENDING ARCHETYPE DETECTION

SPENDING ARCHETYPES

THE SHOPAHOLIC (35.71% on E-commerce)
THE YOLO SPENDER (Savings Rate: -229.34%)


In [124]:
def print_report():                                           # AI-assisted: Helped format and display the final analytical report.

    print("="*70)
    print("                         SpendDNA REPORT")
    print("="*70)

    # =========================================================
    # 1. EXECUTIVE SUMMARY
    # =========================================================

    credits = data[data["Type"] == "credit"]["Amount"].sum()
    debits = data[data["Type"] == "debit"]["Amount"].sum()

    net = credits - debits
    saving_rate = (net / credits) * 100

    print("\n1. EXECUTIVE SUMMARY")
    print("-"*70)

    print(f"Total Credits       : ₹{credits:,.2f}")
    print(f"Total Debits        : ₹{debits:,.2f}")
    print(f"Net Savings         : ₹{net:,.2f}")
    print(f"Savings Rate        : {saving_rate:.2f}%")
    print(f"Transactions        : {len(data)}")
    vendor_column = "vendor_clean" if "vendor_clean" in data.columns else "Vendor"
    print(f"Unique Vendors      : {data[vendor_column].nunique()}")


    # =========================================================
    # 2. TOP CATEGORIES
    # =========================================================

    category_spend = (
        data[data["Type"] == "debit"]
        .groupby("Category")["Amount"]
        .sum()
        .sort_values(ascending=False)
    )

    print("\n2. TOP CATEGORIES")
    print("-"*70)

    print(category_spend.head(5))


    # =========================================================
    # 3. TOP VENDORS
    # =========================================================

    vendor_spend = (
      data[data["Type"] == "debit"]
      .groupby(vendor_column)["Amount"]
      .sum()
      .sort_values(ascending=False)
    )

    print("\n3. TOP VENDORS")
    print("-"*70)

    print(vendor_spend.head(5))


    # =========================================================
    # 4. MONTHLY SPENDING TREND
    # =========================================================

    month_order = [
        "January", "February", "March",
        "April", "May", "June"
    ]

    monthly_report = (
        data[data["Type"] == "debit"]
        .pivot_table(
            values="Amount",
            index="Category",
            columns="Month",
            aggfunc="sum",
            fill_value=0
        )
        .reindex(columns=month_order)
    )

    print("\n4. MONTHLY SPENDING TREND")
    print("-"*70)

    print(monthly_report)


    # =========================================================
    # 5. TIME OF DAY ANALYSIS
    # =========================================================

    time_report = (
        data[data["Type"] == "debit"]
        .groupby(["Category", "Hour"])["Amount"]
        .sum()
        .unstack(fill_value=0)
    )

    print("\n5. TIME OF DAY ANALYSIS")
    print("-"*70)

    print(time_report)

    # Overall peak spending hour
    hourly_spend = (
        data[data["Type"] == "debit"]
        .groupby("Hour")["Amount"]
        .sum()
    )

    peak_hour = hourly_spend.idxmax()

    print(f"\nPeak Spending Hour : {peak_hour:02d}:00")


    # =========================================================
    # 6. TOP ANOMALIES
    # =========================================================

    print("\n6. TOP ANOMALIES")
    print("-"*70)

    # Recalculate anomalies from debit transactions
    debit_data = data[data["Type"] == "debit"].copy()

    mean_amount = (
        debit_data.groupby("Category")["Amount"]
        .transform("mean")
    )

    std_amount = (
        debit_data.groupby("Category")["Amount"]
        .transform("std")
    )

    debit_data["z_score"] = (
        (debit_data["Amount"] - mean_amount)
        / std_amount
    )

    report_anomalies = (
        debit_data[debit_data["z_score"] > 2]
        .sort_values("z_score", ascending=False)
    )

    print(
        report_anomalies[
           ["Date", vendor_column, "Category", "Amount", "z_score"]
        ].head(5)
    )


    # =========================================================
    # 7. SPENDING ARCHETYPES
    # =========================================================

    print("\n7. SPENDING ARCHETYPES")
    print("-"*70)

    total_debit = debits

    food = data[
        data["Category"].isin(
            ["Food Delivery", "Restaurants", "Cafe"]
        )
    ]["Amount"].sum()

    quick = data[
        data["Category"] == "Quick Commerce"
    ]["Amount"].sum()

    ecommerce = data[
        data["Category"] == "E-commerce"
    ]["Amount"].sum()

    investment = data[
        data["Category"] == "Investments"
    ]["Amount"].sum()

    food_pct = (food / total_debit) * 100
    quick_pct = (quick / total_debit) * 100
    ecommerce_pct = (ecommerce / total_debit) * 100
    investment_pct = (investment / total_debit) * 100

    archetypes = []

    if food_pct > 25:
        archetypes.append(
            f"THE FOODIE ({food_pct:.2f}% on food)"
        )

    if quick_pct > 15:
        archetypes.append(
            f"THE QUICK COMMERCE JUNKIE ({quick_pct:.2f}%)"
        )

    if ecommerce_pct > 15:
        archetypes.append(
            f"THE SHOPAHOLIC ({ecommerce_pct:.2f}% on E-commerce)"
        )

    if investment_pct > 15:
        archetypes.append(
            f"THE INVESTOR ({investment_pct:.2f}% on Investments)"
        )

    if saving_rate < 10:
        archetypes.append(
            f"THE YOLO SPENDER (Savings Rate: {saving_rate:.2f}%)"
        )

    if saving_rate >= 40:
        archetypes.append(
            f"THE DISCIPLINED SAVER (Savings Rate: {saving_rate:.2f}%)"
        )

    for archetype in archetypes:
        print("✓", archetype)


    # =========================================================
    # 8. KEY INSIGHTS
    # =========================================================

    print("\n8. KEY INSIGHTS")
    print("-"*70)

    highest_category = category_spend.idxmax()
    highest_category_amount = category_spend.max()

    highest_vendor = vendor_spend.idxmax()
    highest_vendor_amount = vendor_spend.max()

    print(
        f"• Highest spending category : "
        f"{highest_category} "
        f"(₹{highest_category_amount:,.2f})"
    )

    print(
        f"• Highest spending vendor   : "
        f"{highest_vendor} "
        f"(₹{highest_vendor_amount:,.2f})"
    )

    print(
        f"• Total transactions        : {len(data)}"
    )

    print(
        f"• Savings Rate              : {saving_rate:.2f}%"
    )

    print(
        f"• Peak spending hour       : {peak_hour:02d}:00"
    )

    print("\n" + "="*70)
    print("                       END OF REPORT")
    print("="*70)

In [125]:
print_report()

                         SpendDNA REPORT

1. EXECUTIVE SUMMARY
----------------------------------------------------------------------
Total Credits       : ₹509,774.00
Total Debits        : ₹1,678,901.00
Net Savings         : ₹-1,169,127.00
Savings Rate        : -229.34%
Transactions        : 1310
Unique Vendors      : 42

2. TOP CATEGORIES
----------------------------------------------------------------------
Category
E-commerce       599581.0
Investments      248160.0
Food Delivery    150839.0
Restaurants      117737.0
Housing          108000.0
Name: Amount, dtype: float64

3. TOP VENDORS
----------------------------------------------------------------------
vendor_clean
Amazon      328530.0
Zerodha     210000.0
Flipkart    177510.0
Rent        108000.0
Swiggy       95523.0
Name: Amount, dtype: float64

4. MONTHLY SPENDING TREND
----------------------------------------------------------------------
Month              January  February     March    April      May      June
Category   

/tmp/ipykernel_3512/2296077777.py:74: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


# Final Reflection

This project helped me understand how raw transaction data can be cleaned,
standardised and transformed into meaningful financial insights.

The most challenging part was handling inconsistent transaction descriptions
and mapping them to canonical vendors and categories.

Through this project, I practised Pandas, NumPy, data cleaning, grouping,
aggregation, pivot tables, z-score based anomaly detection and conditional
analysis.

The project also showed me how simple data analysis techniques can be used
to identify real-world spending patterns and behavioural trends.